In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
from scipy.sparse.linalg import svds
import warnings
warnings.filterwarnings("ignore")

df = pd.read_csv("ratings.csv")

train, test = train_test_split(df, test_size=0.2, random_state=42)

n_users = df.userId.nunique()
n_items = df.movieId.nunique()

user_mapper = {u:i for i,u in enumerate(df.userId.unique())}
item_mapper = {m:i for i,m in enumerate(df.movieId.unique())}

train["u"] = train.userId.map(user_mapper)
train["i"] = train.movieId.map(item_mapper)

test["u"] = test.userId.map(user_mapper)
test["i"] = test.movieId.map(item_mapper)

R = np.zeros((n_users, n_items))

for row in train.itertuples():
    R[row.u, row.i] = row.rating

k = 20

U, sigma, Vt = svds(R, k=k)

sigma = np.diag(sigma)

R_hat = np.dot(np.dot(U, sigma), Vt)

preds = []
actuals = []

for row in test.itertuples():
    preds.append(R_hat[row.u, row.i])
    actuals.append(row.rating)

rmse = np.sqrt(mean_squared_error(actuals, preds))
mae = mean_absolute_error(actuals, preds)

print(rmse)
print(mae)

def recommend(user_id, n=10):
    uid = user_mapper[user_id]
    scores = R_hat[uid]
    top_items = np.argsort(scores)[::-1][:n]
    return top_items

3.048650797885761
2.830134773349277
